# Libraries

In [ ]:
# Imports and Robot Arm Setup

"General Imports"
import sympy as sp
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

import os
import tensorflow as tf
import keras
from keras import ops
from keras import layers


: 

# Pendulum3Link Class

In [ ]:
class Pendulum3Link:
  "Initialize length and mass values"
  def __init__(self, L1 = 1, L2 = 1, L3 = 1, m1 = 1, m2 = 0.5, m3 = 1, g = 0,
               x0 = [0, 0, 0, 0, 0, 0], final_pos_des = [0,0],
               tspan = (0, 5), fps = 20,
               tau1 = 0, tau2 = 0, tau3 = 0, max_torque = 10.0,
               customController = None):
    self.L1 = L1
    self.L2 = L2
    self.L3 = L3
    self.m1 = m1
    self.m2 = m2
    self.m3 = m3
    self.tau1 = tau1
    self.tau2 = tau2
    self.tau3 = tau3
    self.initial_position = list(x0)
    self.x0 = np.array(x0, dtype=float)
    self.final_position_des = final_pos_des
    self.tspan = tspan
    self.fps = fps
    self.dt = 1.0 / fps
    self.g = g
    self.max_torque = max_torque
    self.max_speed = 15.0                          # Clamp joint velocities to prevent overflow
    self.position_weight = 200       # wp — sharp peak reward near target
    self.position_tol = 5            # kp — controls sharpness of the peak
    self.coarse_weight = 20.0          # wc — linear "get closer" incentive (always active)
    self.energy_weight = 0.02         # we
    self.velocity_weight = 0.1        # wv — penalizes high joint velocities
    self.customController = customController
    self.current_step = 0
    self.velocity_tol = 0.1                       # Tolerance for "zero velocity" done condition

    # PD controller gains for angle-based control
    # The RL agent outputs desired angles; PD converts them to torques:
    #   tau = Kp * (theta_des - theta) - Kd * dtheta
    # Note: Joints have vastly different inertias. Joint 1 moves all 3 masses,
    # so it needs higher Kp and Kd. Joint 3 is light and needs very little.
    self.Kp = np.array([100.0, 50.0, 20.0]) # Proportional gain — stiffness
    self.Kd = np.array([50.0, 22.0, 4.0])   # Derivative gain — critical damping

    # Intialize the equations automatically
    self.formulateEquations()

    # this is for IK
    self.IKSolved = False


  def formulateEquations(self):
    print("Formulating symbolic equations...")
    t = sp.Symbol('t')

    # Generalized coordinates (functions of time)
    th1 = sp.Function('th1')(t)
    th2 = sp.Function('th2')(t)
    th3 = sp.Function('th3')(t)

    q = sp.Matrix([th1, th2, th3])
    dq = q.diff(t)
    ddq = dq.diff(t)

    # Forward Kinematics
    r1 = sp.Matrix([self.L1*sp.sin(th1), -self.L1*sp.cos(th1)])
    r2 = r1 + sp.Matrix([self.L2*sp.sin(th1+th2), -self.L2*sp.cos(th1+th2)])
    r3 = r2 + sp.Matrix([self.L3*sp.sin(th1+th2+th3), -self.L3*sp.cos(th1+th2+th3)])

    # Mass velocities in Cartesian space
    dr1 = r1.diff(t)
    dr2 = r2.diff(t)
    dr3 = r3.diff(t)

    # Energy
    T = 0.5 * self.m1 * dr1.dot(dr1) + 0.5 * self.m2 * dr2.dot(dr2) + 0.5 * self.m3 * dr3.dot(dr3)
    V = self.m1 * self.g * r1[1] + self.m2 * self.g * r2[1] + self.m3 * self.g * r3[1]
    Lagrangian = T - V

    # Euler-Lagrange Equations
    dL_dq = sp.Matrix([Lagrangian.diff(qi) for qi in q])
    dL_ddq = sp.Matrix([Lagrangian.diff(dqi) for dqi in dq])
    ddt_dL_ddq = dL_ddq.diff(t)
    EL_LHS = ddt_dL_ddq - dL_dq

    # Create static symbols for lambdification
    th1_, th2_, th3_ = sp.symbols('th1_ th2_ th3_')
    dth1_, dth2_, dth3_ = sp.symbols('dth1_ dth2_ dth3_')
    ddth1_, ddth2_, ddth3_ = sp.symbols('ddth1_ ddth2_ ddth3_')
    tau1, tau2, tau3 = sp.symbols('tau1 tau2 tau3')

    q_ = sp.Matrix([th1_, th2_, th3_])
    dq_ = sp.Matrix([dth1_, dth2_, dth3_])
    ddq_ = sp.Matrix([ddth1_, ddth2_, ddth3_])
    tau_ = sp.Matrix([tau1, tau2, tau3])

    # Dictionary substitution from highest order derivatives to lowest
    sub_dict = {
        ddq[0]: ddq_[0], ddq[1]: ddq_[1], ddq[2]: ddq_[2],
        dq[0]: dq_[0], dq[1]: dq_[1], dq[2]: dq_[2],
        q[0]: q_[0], q[1]: q_[1], q[2]: q_[2]
    }

    EL_LHS_ = EL_LHS.subs(sub_dict)

    # Inertial matrix M(q) and other variables
    M = EL_LHS_.jacobian(ddq_)
    f1 = -EL_LHS_.subs({ddth1_: 0, ddth2_: 0, ddth3_: 0})

    # CRITICAL PHYSICS BUG FIX:
    # Because th2 and th3 are RELATIVE angles (kinematics use th1+th2), the virtual work
    # done by the joint motors is exactly tau * delta_th.
    # The previous code (tau1-tau2) erroneously assumed th1, th2, th3 were ABSOLUTE angles.
    # Applying tau3 was magically pushing the second link backwards in the math!
    f2 = sp.Matrix([tau1, tau2, tau3])
    f = f1 + f2

    B = f2.jacobian(tau_)
    B_mat = np.array(B).astype(np.float64)

    M_func = sp.lambdify((q_,), M, "numpy")
    f_func = sp.lambdify((q_, dq_, tau_), f, "numpy")
    f1_func = sp.lambdify((q_, dq_), f1, "numpy")

    self.M_func = M_func
    self.f_func = f_func
    self.f1_func = f1_func
    self.formulatedBool = True

    print("Done!")


  def setInputs(self, action):
    self.tau1 = action[0]
    self.tau2 = action[1]
    self.tau3 = action[2]

  def pd_control(self, theta_des):
    """
    PD controller: converts desired joint angles into torques.
      tau = Kp * (shortest_angular_error) - Kd * dtheta
    Torques are clamped to [-max_torque, max_torque].
    """
    theta = self.x0[0:3]     # current joint angles
    dtheta = self.x0[3:6]    # current joint velocities

    # Calculate shortest angular distance to prevent winding bug.
    # If theta reaches 4*pi, and theta_des is 0, the error should be 0, not -4*pi!
    error = (theta_des - theta + np.pi) % (2 * np.pi) - np.pi

    tau = self.Kp * error - self.Kd * dtheta
    tau = np.clip(tau, -self.max_torque, self.max_torque)
    return tau

  def tau_controller(self, x, t):
    """
    Returns the current torques for use in dynamics().
    x is the state vector [th1, th2, th3, dth1, dth2, dth3]
    """
    return np.array([self.tau1, self.tau2, self.tau3])


  def get_full_state(self, x, tau_input):
    """
    Given a state x and 3 input torques, outputs the full kinematic state:
    Returns: (positions, velocities, accelerations)
    """
    q_val = x[0:3]
    dq_val = x[3:6]

    # M(q) * ddq = f(q, dq, tau)  => ddq = inv(M) * f
    M_val = self.M_func(q_val)
    f_val = self.f_func(q_val, dq_val, tau_input).flatten()

    # Solve linear system for accelerations
    ddq_val = np.linalg.solve(M_val, f_val)

    return q_val, dq_val, ddq_val

  def dynamics(self, t_val, x):
    """
    ODE function for scipy.integrate.solve_ivp
    dx/dt = [dq, ddq]
    """
    tau_input = self.tau_controller(x, t_val)

    _, dq_val, ddq_val = self.get_full_state(x, tau_input)
    return np.concatenate((dq_val, ddq_val))

  def runSimulation(self):

    t_eval = np.linspace(self.tspan[0], self.tspan[1], int((self.tspan[1]-self.tspan[0])*self.fps))
    sol = solve_ivp(self.dynamics, self.tspan, self.x0, method='RK45', t_eval=t_eval)
    self.tout = sol.t
    self.xout = sol.y.T

  def reset(self):
    """Reset pendulum to initial position for a new episode."""
    self.x0 = np.array(self.initial_position, dtype=float)
    self.xout = self.x0.reshape(1, -1)   # Needed for endPosition Calculation
    self.current_step = 0
    self.tau1 = 0
    self.tau2 = 0
    self.tau3 = 0
    return self.get_state()

  def get_state(self):
    """
    Returns the RL state: [th1, th2, th3, dth1, dth2, dth3, error_x, error_y]
    Adding Cartesian error gives the agent crucial spatial context.
    """
    # Force endPosition to use the current x0
    end_pos = self.endPosition(state=self.x0)
    error_x = self.final_position_des[0] - end_pos[0]
    error_y = self.final_position_des[1] - end_pos[1]
    return np.concatenate((self.x0, [error_x, error_y]))

  def step(self, action):
    """
    Advance simulation by one timestep dt using a direct RK4 step.

    Action is interpreted as DESIRED JOINT ANGLES (not raw torques).
    A PD controller converts desired angles to torques internally:
      tau = Kp * (theta_des - theta) - Kd * dtheta

    Returns: (new_state, reward, done)
    """
    # PD control: convert desired angles -> torques
    tau = self.pd_control(action)
    self.setInputs(tau)

    # RK4 integration (4th-order Runge-Kutta, single step)
    # NOTE: Intermediate states must also have clamped velocities,
    # otherwise the dth^2 terms in the dynamics overflow during sub-steps.
    t = self.current_step * self.dt
    x = self.x0
    dt = self.dt

    k1 = np.asarray(self.dynamics(t, x), dtype=float)

    x2 = x + dt/2 * k1
    x2[3:6] = np.clip(x2[3:6], -self.max_speed, self.max_speed)
    k2 = np.asarray(self.dynamics(t + dt/2, x2), dtype=float)

    x3 = x + dt/2 * k2
    x3[3:6] = np.clip(x3[3:6], -self.max_speed, self.max_speed)
    k3 = np.asarray(self.dynamics(t + dt/2, x3), dtype=float)

    x4 = x + dt * k3
    x4[3:6] = np.clip(x4[3:6], -self.max_speed, self.max_speed)
    k4 = np.asarray(self.dynamics(t + dt, x4), dtype=float)

    self.x0 = x + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    # Clamp velocities on final state too
    self.x0[3:6] = np.clip(self.x0[3:6], -self.max_speed, self.max_speed)

    self.xout = self.x0.reshape(1, -1)   # Store as (1, 6) for endPosition compatibility
    self.current_step += 1

    reward_val = self.reward()

    # Done condition: end-effector is close to target with near-zero velocity
    #end_pos = self.endPosition()
    #pos_error = (end_pos[0] - self.final_position_des[0])**2 + \
    #            (end_pos[1] - self.final_position_des[1])**2
    #velocities = self.x0[3:6]
    #speed = np.linalg.norm(velocities)
    #done = (pos_error < 0.01) and (speed < self.velocity_tol)

    done = False

    return self.get_state(), reward_val, done


  def endPosition(self, state=None):
    """
    Compute the end-effector (tip of link 3) position.
    Bug #4: Previously used list concatenation instead of vector addition.
    Now uses np.array for proper element-wise math.
    """
    if state is not None:
      th1 = state[0]
      th2 = state[1]
      th3 = state[2]
    elif hasattr(self, 'xout') and self.xout is not None:
      th1 = self.xout[-1, 0]
      th2 = self.xout[-1, 1]
      th3 = self.xout[-1, 2]
    else:
      th1 = self.x0[0]
      th2 = self.x0[1]
      th3 = self.x0[2]

    r1 = np.array([self.L1*np.sin(th1), -self.L1*np.cos(th1)])
    r2 = r1 + np.array([self.L2*np.sin(th1+th2), -self.L2*np.cos(th1+th2)])
    r3 = r2 + np.array([self.L3*np.sin(th1+th2+th3), -self.L3*np.cos(th1+th2+th3)])

    self.end_th1 = th1
    self.end_th2 = th2
    self.end_th3 = th3

    return r3



  import numpy as np

  def solve_ik(self, x_des, y_des):
      """
      phi_des: The desired absolute angle of the 3rd link (in radians)
      Returns th1, th2, th3 instantly.
      """

      phi_des = np.pi / 2
      L1, L2, L3 = self.L1, self.L2, self.L3

      # 1. Calculate the Wrist Position (xw, yw)
      # The wrist is just the end-effector minus the 3rd link's contribution
      # Based on your coordinate system (x=sin, y=-cos):
      xw = x_des - L3 * np.sin(phi_des)
      yw = y_des + L3 * np.cos(phi_des)

      # 2. Solve 2-Link IK for the Wrist (L1 and L2)
      dist_sq = xw**2 + yw**2
      dist = np.sqrt(dist_sq)

      # Reachability check
      if dist > (L1 + L2) or dist < abs(L1 - L2):
          return None, None, None

      # Law of Cosines for th2
      cos_th2 = (dist_sq - L1**2 - L2**2) / (2 * L1 * L2)
      cos_th2 = np.clip(cos_th2, -1.0, 1.0)

      # ELBOW DOWN: Force positive th2
      th2 = np.arccos(cos_th2)

      # Solve for th1
      # Standard 2-link geometry adjusted for your x=sin/y=-cos setup
      alpha = np.arctan2(xw, -yw) # Angle to the wrist
      beta = np.arctan2(L2 * np.sin(th2), L1 + L2 * np.cos(th2)) # Interior angle
      th1 = alpha - beta

      # 3. Calculate th3
      # Since phi = th1 + th2 + th3
      th3 = phi_des - th1 - th2

      return th1, th2, th3



  def reward(self):

    final_position = self.endPosition()

    x = final_position[0]
    y = final_position[1]

    x_des = self.final_position_des[0]
    y_des = self.final_position_des[1]

    if self.IKSolved == False:
      self.IKSolved = True
      # Usage:
      self.th1_des, self.th2_des, self.th3_des = self.solve_ik(x_des=x_des, y_des=y_des)

    th1 = self.end_th1
    th2 = self.end_th2
    th3 = self.end_th3

    # JUST CHECKING
    # r1 = np.array([self.L1*np.sin(self.th1_des), -self.L1*np.cos(self.th1_des)])
    # r2 = r1 + np.array([self.L2*np.sin(self.th1_des+self.th2_des), -self.L2*np.cos(self.th1_des+self.th2_des)])
    # r3 = r2 + np.array([self.L3*np.sin(self.th1_des+self.th2_des+self.th3_des), -self.L3*np.cos(self.th1_des+self.th2_des+self.th3_des)])

    # print(r3)
    # print(self.final_position_des)

    # breakpoint()
    # END CHECKING

    error_position = np.sqrt((th1 - self.th1_des)**2 + (th2 - self.th2_des)**2 + (th3 - self.th3_des)**2)
    energy_penalty = np.sqrt(self.tau1**2 + self.tau2**2 + self.tau3**2)
    velocity_penalty = np.linalg.norm(self.x0[3:6])  # L2 norm of joint velocities

    self.position_weight = 2.0       # wp — sharp peak reward near target
    self.position_tol = 5            # kp — controls sharpness of the peak
    self.coarse_weight = 0.2          # wc — linear "get closer" incentive (always active)
    self.energy_weight = 0.0002         # we
    self.velocity_weight = 0.001        # wv — penalizes high joint velocities

    wp = 2.0 #self.position_weight
    kp = 5 # self.position_tol
    wc = 0.2 #self.coarse_weight
    we = 0.0002 #self.energy_weight
    wv = 0.001 #self.velocity_weight

    # Two-term position reward:
    #   1. COARSE: -wc * distance — always pushes arm toward target,
    #      provides gradient even when far away (unlike exp which saturates to 0)
    #   2. FINE:   wp * exp(-kp * distance) — sharp peak near target,
    #      only activates within ~1/kp meters, incentivizes precision
    coarse_reward = wc * np.exp(-error_position)
    fine_reward = wp * np.exp(-kp * error_position)

    reward = fine_reward + coarse_reward - we * energy_penalty - wv * velocity_penalty

    return reward


  def plotResults(self, desBool=None):
    print("Only trust results if runSimulation has been ran beforehand")
    plt.figure(figsize=(8, 4))
    plt.hlines(y = [self.th1_des, self.th2_des, self.th3_des], colors = ['k', 'b', 'r'], linestyles = [':',':',':'], xmin= 0, xmax= self.tout[-1])
    plt.plot(self.tout, self.xout[:, 0], 'k-', label='Theta 1')
    plt.plot(self.tout, self.xout[:, 1], 'b-', label='Theta 2')
    plt.plot(self.tout, self.xout[:, 2], 'r-', label='Theta 3')
    plt.xlabel('Time (s)')
    plt.ylabel('Angle (rad)')
    plt.title('Triple Pendulum Angles Over Time')
    plt.legend()
    plt.grid(True)
    plt.show()




  def run_evaluation(self, model, max_torque, max_steps):
    """Run a deterministic evaluation episode (mean actions, no noise) and store trajectory."""
    state = self.reset()
    states = [state.copy()]

    for _ in range(max_steps):
      state_tensor = ops.expand_dims(
          ops.convert_to_tensor(state, dtype="float32"), 0
      )
      # Deterministic: use mean action directly (no noise)
      # Output is desired angles, scaled to [-pi, pi]
      actor_mean_out = model(state_tensor)
      action = np.clip(actor_mean_out.numpy()[0] * np.pi, -np.pi, np.pi)

      state, _, done = self.step(action)
      states.append(state.copy())
      if done:
        break

    # Store trajectory so animateResults can use it
    self.xout = np.array(states)
    self.tout = np.arange(len(states)) * self.dt



  def animateResults(self):
      print("Only trust results if runSimulation has been ran beforehand")
      print("Generating animation (this may take a moment)...")
      fig, ax = plt.subplots(figsize=(6, 6))

      bound = (self.L1 + self.L2 + self.L3) * 1.2
      ax.set_xlim(-bound, bound)
      ax.set_ylim(-bound, bound)
      ax.set_aspect('equal')
      ax.grid(True)
      ax.set_title("Triple Pendulum Animation")


      # Show target position
      target = self.final_position_des
      ax.plot(target[0], target[1], 'g*', markersize=15, label='Target')
      ax.legend()


      line, = ax.plot([], [], 'k-', lw=2)
      masses, = ax.plot([], [], 'ro', markersize=10, markerfacecolor='r')

      def init():
          line.set_data([], [])
          masses.set_data([], [])
          return line, masses

      def update(i):
          th1_i = self.xout[i, 0]
          th2_i = self.xout[i, 1]
          th3_i = self.xout[i, 2]

          x1 = self.L1 * np.sin(th1_i)
          y1 = -self.L1 * np.cos(th1_i)
          x2 = x1 + self.L2 * np.sin(th1_i + th2_i)
          y2 = y1 - self.L2 * np.cos(th1_i + th2_i)
          x3 = x2 + self.L3 * np.sin(th1_i + th2_i + th3_i)
          y3 = y2 - self.L3 * np.cos(th1_i + th2_i + th3_i)

          xs = [0, x1, x2, x3]
          ys = [0, y1, y2, y3]

          line.set_data(xs, ys)
          masses.set_data(xs[1:], ys[1:])
          return line, masses

      num_frames = len(self.xout)
      anim = FuncAnimation(fig, update, frames=num_frames, init_func=init,
                          blit=True, interval=1000/self.fps)

      plt.close(fig) # Prevent static duplicate plot in Colab
      print("Done!")
      display(HTML(anim.to_jshtml()))

# Reinforcement Learning

In [ ]:
# Reinforcement Learning Code

"Initialize an instance"
pend = Pendulum3Link(
    x0 = [np.pi/2, 0, 0, 0, 0, 0],
    final_pos_des = [1.5, -1.5],
    tspan = (0.0, 5.0),
    fps = 20,
    max_torque = 50.0
)

# Reinforcement learning hyperparameters
gamma = 0.99                              # Discount factor (recall the Bellman equation)
gae_lambda = 0.95                         # GAE lambda
clip_ratio = 0.2                          # PPO clipping parameter
batch_size = 2000                         # PPO rollout batch size (in steps)
minibatch_size = 64                       # SGD minibatch size
epochs = 10                               # PPO training epochs per batch
eps = 1e-8
max_steps_per_episode = 200
max_torque = pend.max_torque

num_inputs = 8    # th1, th2, th3, dth1, dth2, dth3, err_x, err_y
num_actions = 3   # desired angles for each of the 3 joints
num_hidden_1 = 64   # first hidden layer width
num_hidden_2 = 64   # second hidden layer width

# PPO Agent

In [ ]:
# PPO Agent Definition
class PPOAgent:
    def __init__(self, num_inputs, num_actions):
        self.actor_model = self.build_actor(num_inputs, num_actions)
        self.critic_model = self.build_critic(num_inputs)

        # State-independent log_std (crucial fix for convergence)
        self.log_std = tf.Variable(initial_value=-1.0 * tf.ones(num_actions, dtype=tf.float32), trainable=True, name="actor_log_std")

        self.actor_optimizer = keras.optimizers.Adam(learning_rate=3e-4, global_clipnorm=1.0)
        self.critic_optimizer = keras.optimizers.Adam(learning_rate=1e-3, global_clipnorm=1.0)
        self.clip_ratio = clip_ratio
        self.entropy_coef = 0.005 # Small exploration bonus

    def build_actor(self, num_in, num_out):
        inputs = layers.Input(shape=(num_in,))
        x = layers.Dense(64, activation="swish")(inputs)
        x = layers.Dense(64, activation="swish")(x)
        mean = layers.Dense(num_out, activation="tanh", name="actor_mean")(x)
        return keras.Model(inputs=inputs, outputs=mean)

    def build_critic(self, num_in):
        inputs = layers.Input(shape=(num_in,))
        x = layers.Dense(64, activation="swish")(inputs)
        x = layers.Dense(64, activation="swish")(x)
        value = layers.Dense(1, name="critic_value")(x)
        return keras.Model(inputs=inputs, outputs=value)

    def get_action(self, state):
        state_tensor = tf.expand_dims(tf.convert_to_tensor(state, dtype=tf.float32), 0)

        # Forward pass
        mean = self.actor_model(state_tensor)[0]
        std = tf.exp(tf.clip_by_value(self.log_std, -5.0, 2.0))

        # Reparameterization
        noise = tf.random.normal(shape=tf.shape(mean))
        action_raw = mean + std * noise

        # Action execution logic
        action = tf.clip_by_value(action_raw * np.pi, -np.pi, np.pi)

        # Log probability
        log_prob = -0.5 * tf.reduce_sum(
            ((action_raw - mean) / (std + 1e-8)) ** 2
            + 2.0 * self.log_std
            + tf.math.log(2.0 * np.pi)
        )

        # Value
        value = self.critic_model(state_tensor)[0, 0]

        return action.numpy(), action_raw.numpy(), log_prob.numpy(), value.numpy()

    def train_step(self, states, actions_raw, old_log_probs, advantages, returns, epochs, batch_size):
        # Convert all to tensors
        states = tf.convert_to_tensor(states, dtype=tf.float32)
        actions_raw = tf.convert_to_tensor(actions_raw, dtype=tf.float32)
        old_log_probs = tf.convert_to_tensor(old_log_probs, dtype=tf.float32)
        advantages = tf.convert_to_tensor(advantages, dtype=tf.float32)
        returns = tf.convert_to_tensor(returns, dtype=tf.float32)

        dataset = tf.data.Dataset.from_tensor_slices((states, actions_raw, old_log_probs, advantages, returns))
        dataset = dataset.shuffle(buffer_size=len(states)).batch(batch_size)

        actor_losses = []
        critic_losses = []

        for _ in range(epochs):
            for mb_s, mb_a, mb_old_log_p, mb_adv, mb_ret in dataset:
                with tf.GradientTape() as actor_tape, tf.GradientTape() as critic_tape:
                    # Current policy
                    mean = self.actor_model(mb_s)
                    std = tf.exp(tf.clip_by_value(self.log_std, -5.0, 2.0))

                    # Compute new log probs
                    new_log_probs = -0.5 * tf.reduce_sum(
                        ((mb_a - mean) / (std + 1e-8)) ** 2
                        + 2.0 * self.log_std
                        + tf.math.log(2.0 * np.pi),
                        axis=-1
                    )

                    # PPO Surrogate Objective
                    ratio = tf.exp(new_log_probs - mb_old_log_p)
                    clipped_ratio = tf.clip_by_value(ratio, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio)
                    surrogate1 = ratio * mb_adv
                    surrogate2 = clipped_ratio * mb_adv

                    # Actor Loss (negative because we minimize in TF)
                    entropy_bonus = tf.reduce_sum(self.log_std + 0.5 * tf.math.log(2.0 * np.pi * np.e))
                    actor_loss = -tf.reduce_mean(tf.minimum(surrogate1, surrogate2)) - self.entropy_coef * entropy_bonus

                    # Critic Loss (MSE)
                    values = tf.squeeze(self.critic_model(mb_s), axis=-1)
                    critic_loss = keras.losses.mean_squared_error(mb_ret, values)

                # Update Actor
                actor_grads = actor_tape.gradient(actor_loss, self.actor_model.trainable_variables + [self.log_std])
                self.actor_optimizer.apply_gradients(zip(actor_grads, self.actor_model.trainable_variables + [self.log_std]))

                # Update Critic
                critic_grads = critic_tape.gradient(critic_loss, self.critic_model.trainable_variables)
                self.critic_optimizer.apply_gradients(zip(critic_grads, self.critic_model.trainable_variables))

                actor_losses.append(actor_loss.numpy())
                critic_losses.append(critic_loss.numpy())

        return np.mean(actor_losses), np.mean(critic_losses)

# Helper Functions

In [ ]:
def calculate_gae(rewards, values, gamma, lam):
    advantages = np.zeros_like(rewards, dtype=np.float32)
    last_adv = 0
    t_end = len(rewards)
    for t in reversed(range(t_end)):
        if t == t_end - 1:
            next_value = 0.0 # Assuming horizon termination
        else:
            next_value = values[t+1]

        delta = rewards[t] + gamma * next_value - values[t]
        advantages[t] = last_adv = delta + gamma * lam * last_adv

    returns = advantages + np.array(values, dtype=np.float32)
    return advantages.tolist(), returns.tolist()

def trainModel(agent, pend, TARGET_REWARD, max_iter = -1):
  # Initialization
  # agent = PPOAgent(num_inputs, num_actions)

  print("Starting PPO Training...")
  batch_num = 0
  running_reward = 0

  while True:  # Run until solved
      # Buffer structures
      b_states, b_actions, b_log_probs, b_advantages, b_returns = [], [], [], [], []

      steps_collected = 0
      total_batch_reward = 0
      episodes_in_batch = 0

      # 1. Rollout Phase
      while steps_collected < batch_size:
          state = pend.reset()
          episode_reward = 0

          ep_states, ep_actions, ep_rewards, ep_values, ep_log_probs = [], [], [], [], []

          for _ in range(max_steps_per_episode):
              action, action_raw, log_prob, value = agent.get_action(state)
              next_state, reward, done = pend.step(action)

              # Record transition
              ep_states.append(state)
              ep_actions.append(action_raw)
              ep_rewards.append(reward)
              ep_values.append(value)
              ep_log_probs.append(log_prob)

              state = next_state
              episode_reward += reward
              steps_collected += 1

              if done or np.isnan(reward) or np.any(np.isnan(state)):
                  break

          # Calculate GAE
          adv, ret = calculate_gae(ep_rewards, ep_values, gamma, gae_lambda)

          b_states.extend(ep_states)
          b_actions.extend(ep_actions)
          b_log_probs.extend(ep_log_probs)
          b_advantages.extend(adv)
          b_returns.extend(ret)

          total_batch_reward += episode_reward
          episodes_in_batch += 1

      mean_ep_reward = total_batch_reward / episodes_in_batch
      running_reward = 0.05 * mean_ep_reward + 0.95 * running_reward if running_reward != 0 else mean_ep_reward

      # Normalize advantages
      adv_arr = np.array(b_advantages, dtype=np.float32)
      adv_arr = (adv_arr - np.mean(adv_arr)) / (np.std(adv_arr) + 1e-8)

      # 2. Optimization Phase
      aloss, closs = agent.train_step(
          b_states, b_actions, b_log_probs, adv_arr, b_returns,
          epochs=epochs, batch_size=minibatch_size
      )

      batch_num += 1
      template = "Batch {} | Running Reward: {:.2f} | Latest Mean Reward: {:.2f} | Actor Loss: {:.4f} | Critic Loss: {:.4f}"
      print(template.format(batch_num, running_reward, mean_ep_reward, aloss, closs))

      # Show animation every 10 batches
      if batch_num % 10 == 0:
          pend.run_evaluation(agent.actor_model, max_torque, max_steps_per_episode)
          # pend.animateResults()

      if running_reward > TARGET_REWARD:
          print(f"Solved at Batch {batch_num}!")
          pend.run_evaluation(agent.actor_model, max_torque, max_steps_per_episode)
          # pend.animateResults()
          return agent, running_reward, pend, batch_num
          break
      elif (batch_num >= max_iter and max_iter >0):
          print(f"Quit at Batch {batch_num}!")
          pend.run_evaluation(agent.actor_model, max_torque, max_steps_per_episode)
          # pend.animateResults()
          return agent, running_reward, pend, batch_num
          break;

# Run PPO normally

In [ ]:
agent = PPOAgent(num_inputs, num_actions)

target_reward = 200
max_iter = 75

"Train the model"
trained_agent, running_reward, trained_pendulum, batch_num = trainModel(agent, pend, target_reward, max_iter)

In [ ]:
trained_pendulum.animateResults()

# Plot results

In [ ]:
trained_pendulum.plotResults()

In [ ]:
import dill

with open('trained_agent.pkl', 'wb') as f:
    dill.dump(trained_agent, f)

with open('trained_pendulum.pkl', 'wb') as f:
    dill.dump(trained_pendulum, f)

